# Linear Regression, end to end — Cars.csv (10-step ML pipeline)

This notebook follows the **10-step ML pipeline** from the course diagram:

`Business Understanding → Data Collection → Data Understanding → Data Preparation → Model Building → Model Training → Model Testing → Model Evaluation → Model Deployment → CI/CD`

Same underlying work as any ML project — this is just the more granular, industry-standard version of the 7-step flow used in `03-Logistic Regression/scripting.ipynb`. Refer back to `02-Linear Regression/theory.md` for the math (the `y = mx + b` formula, MSE, OLS, Gradient Descent) behind what `sklearn` does automatically in Step 6 below.

---
## 1. Business Understanding

**Dataset:** `assets/Cars.csv` — 81 cars, with `HP` (horsepower), `MPG` (miles per gallon), `VOL` (volume, likely of the car body/trunk in some standard unit), `SP` (top speed), `WT` (weight).

**The business question:** *given a car's horsepower, volume, top speed, and weight, can we predict its fuel efficiency (MPG)?*

Why this matters in the real world: a car manufacturer designing a new model wants to know, before ever building a physical prototype, roughly what fuel efficiency a given combination of specs will produce. Being able to predict MPG from design specs lets engineers make tradeoffs early (e.g. "if we increase horsepower by X, how much MPG do we lose, and is that worth it?") instead of only finding out after expensive physical testing.

**Why Linear Regression specifically:** MPG is a **number** (continuous), not a category — recall Chapter 1: numeric target → Regression, not Classification. And there's no reason yet to assume the relationship is anything other than roughly linear, so Linear Regression (Chapter 2) is the natural first model to try (recall Chapter 1's recap, Step 6: "start simple, then get fancy").

---
## 2. Data Collection

For this notebook, data collection is already done for us — `Cars.csv` is provided directly. In a real project, this step would involve pulling from a database, an API, sensor logs, or a manufacturer's internal specs sheet, and would need its own checks (is this data even trustworthy? is it representative of what we'll see in production?).

In [ ]:
import pandas as pd   # pandas: the standard library for tabular data (rows/columns) -- gives us the DataFrame object
import numpy as np    # numpy: fast numeric array operations -- pandas is built on top of it, and we use it directly for things like np.sqrt()

df = pd.read_csv("../assets/Cars.csv")   # pd.read_csv() reads a CSV file straight into a DataFrame (a table, like an Excel sheet in code)
df.head()                                 # .head() shows the first 5 rows -- a quick, cheap sanity check on what the data actually looks like


---
## 3. Data Understanding

Same spirit as Chapter 1/3: before touching a model, understand what you actually have — shape, types, missing values, summary stats, and how features relate to each other and to the target.

In [ ]:
print(df.shape)   # .shape is a (rows, columns) tuple -- (81, 5) -- a small dataset, worth keeping in mind for how much we can trust the results
df.info()         # .info() prints column names, dtypes (int64/float64/object), and non-null counts all in one summary


**Reading `df.info()`:** all 5 columns are non-null (0 missing values) and numeric (`int64`/`float64`) already — unlike `claim.csv` in Chapter 3, there's no missing-data cleanup needed here. That's not always true in real projects; this dataset just happens to be clean.

In [ ]:
df.describe()   # .describe() gives count/mean/std/min/25%/50%/75%/max for every numeric column -- a fast statistical summary of the whole table


In [ ]:
df.corr()
# .corr() computes the PAIRWISE correlation between every pair of numeric columns, as one matrix, all at once.
# Each value ranges -1 to +1: +1 = perfectly move together, -1 = perfectly move opposite, 0 = no linear relationship.
# THIS IS THE MOST IMPORTANT CELL IN THIS STEP -- watch VOL vs WT and HP vs SP below.


**Reading the correlation matrix — a genuine red flag before we even build anything:**

- `VOL` and `WT` correlate at **0.999** — essentially the same information, twice, under two different column names.
- `HP` and `SP` correlate at **0.974** — also nearly redundant.

This is exactly the **multicollinearity** warning from `02-Linear Regression/theory.md` Section 8 ("input features shouldn't be near-duplicates of each other, e.g. `length_cm` and `length_inches`"). We'll keep all 4 features for this pass (to actually *see* the effect multicollinearity has on the model, in Step 8's interpretation), but flag now that this is something a real project would address — e.g. by dropping one of each near-duplicate pair, or combining them.

In [ ]:
import matplotlib.pyplot as plt   # matplotlib: the base Python plotting library -- everything else (including seaborn) draws on top of it
import seaborn as sns              # seaborn: a wrapper around matplotlib with nicer defaults and built-in statistical plot types

sns.set_style("whitegrid")   # cosmetic only -- adds light gridlines to every plot drawn from here on, makes values easier to read off

fig, axes = plt.subplots(2, 2, figsize=(10, 8))   # plt.subplots(2, 2, ...) creates one figure containing a 2x2 GRID of separate mini-plots (axes),
                                                    # so we can compare all 4 features against MPG side by side instead of one plot at a time
for ax, col in zip(axes.flat, ["HP", "VOL", "SP", "WT"]):
    # axes.flat turns the 2x2 grid of axes into a flat list of 4, so we can loop over them one at a time alongside the 4 column names
    sns.scatterplot(data=df, x=col, y="MPG", ax=ax)   # scatterplot: one dot per row, showing this feature (x) against MPG (y) -- ax= tells it WHICH mini-plot to draw into
    ax.set_title(f"MPG vs {col}")                      # .set_title() labels that specific mini-plot
plt.tight_layout()   # .tight_layout() auto-adjusts spacing so titles/labels don't overlap between the 4 mini-plots
plt.show()            # .show() actually renders/displays the figure


**Reading these plots:** every feature shows a visible **downward** trend against MPG on its own — more horsepower, more volume, higher top speed, more weight all individually associate with *lower* MPG. Keep this in mind for Step 8 — we're about to see this simple, expected pattern get more complicated once all 4 features are combined into one model.

---
## 4. Data Preparation

No missing values to handle here (Step 3 confirmed that), so preparation is lighter than in Chapter 3's `claim.csv` notebook. What's still needed: separate features from target, and split into train/test sets.

In [ ]:
from sklearn.model_selection import train_test_split
# train_test_split: sklearn's standard function to randomly split a dataset into a TRAINING portion (the model learns from this)
# and a TESTING portion (held back, never trained on -- used only to check the model generalizes to new data, not just memorized).

X = df.drop(columns=["MPG"])   # .drop(columns=[...]) returns a NEW DataFrame with those columns removed -- X = features: HP, VOL, SP, WT
y = df["MPG"]                   # selecting a single column by name returns a Series (a single column of values) -- y = target: MPG (a continuous number -> regression)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # hold back 20% of rows for evaluation only (Step 7-8); the remaining 80% is used for training
    random_state=42,    # a fixed "seed" for the random split -- same number in, same exact split out every time we rerun this cell (reproducibility)
)
# train_test_split() ALWAYS returns exactly 4 things in this order: X_train, X_test, y_train, y_test -- features/target x train/test

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
# Train: (64, 4), Test: (17, 4)


**Note:** unlike Chapter 3's classification notebook, there's no `stratify` here — stratifying is a classification concept (keeping class *ratios* consistent across splits). For a regression target like MPG, there's no "class ratio" to preserve; a plain random split is the standard approach. With only 81 rows total (64 for training), also worth flagging: this is a genuinely small dataset, so results here should be read as illustrative, not as a production-grade model — a real project would want far more than 81 cars before trusting this.

---
## 5. Model Building

This step is choosing *and instantiating* the algorithm — deciding what kind of model this will be, before any data has touched it. Recall Chapter 1: since we've established this is Supervised + Regression + roughly-linear (Step 1), **Linear Regression** is the model choice, same reasoning as Chapter 1's recap checklist.

In [ ]:
from sklearn.linear_model import LinearRegression
# LinearRegression: sklearn's implementation of the y = mx + b algorithm from theory.md -- fits multiple features at once via Ordinary Least Squares.

model = LinearRegression()
# At this point the model exists but knows NOTHING yet -- no m's, no b.
# "Model Building" is just this decision + setup step, separate from "Model Training" (Step 6) where it actually learns.


---
## 6. Model Training

This is where `02-Linear Regression/theory.md` becomes real code. `sklearn`'s `LinearRegression` does exactly what the theory describes: finds the `m` for each feature and the single `b` that minimize Mean Squared Error across the training data, using Ordinary Least Squares (the direct-solve method, theory.md Section 5) — no iteration needed, unlike Logistic Regression in Chapter 3.

In [ ]:
model.fit(X_train, y_train)
# .fit(X, y) is THE training call for every sklearn model -- always takes (features, target) and mutates the model object in place
# (nothing is returned to capture -- the learned m's and b now live INSIDE `model`). This IS the "learning": OLS solves directly for the best m's and b.

for feature, m in zip(X.columns, model.coef_):
    # model.coef_ is an array of the learned slopes (m's), one per feature, in the SAME ORDER as the training columns (X.columns)
    # zip() pairs each column name up with its matching learned slope so we can print them together
    print(f"{feature:>5}: m = {m:+.4f}")
print(f"{'b':>5}: b = {model.intercept_:+.4f}")   # model.intercept_ is the single learned b (bias/intercept) -- always a lone number, not an array, for one target
# HP: m = -0.2205
# VOL: m = -0.5397
# SP: m = +0.5336
# WT: m = +0.9453
# b: b = +17.7257


---
## 7. Model Testing

Run the trained model on the **held-out test set** — data it has never seen — and get its predictions. This step is just *generating* predictions; judging whether they're good comes next in Step 8 (Evaluation). Keeping these as two separate steps (as the 10-step pipeline does) is a useful discipline: it forces you to look at actual predictions before jumping straight to a summary metric.

In [ ]:
y_pred = model.predict(X_test)
# .predict(X) is THE prediction call for every sklearn model -- feed it features (no y needed, since it doesn't know the answer),
# get back an array of predicted values, one per row, in the same order as X_test.

comparison = pd.DataFrame({
    "actual_mpg": y_test.values,     # .values converts the y_test Series into a plain numpy array (matches y_pred's array type, for a clean side-by-side table)
    "predicted_mpg": y_pred,
    "error": y_test.values - y_pred,  # actual minus predicted -- this is the RESIDUAL from theory.md, computed per row
})
comparison.round(2)   # .round(2) just rounds every number in the table to 2 decimal places for readability -- doesn't change the underlying data


**Reading this table:** each row is one car from the test set the model never trained on. `error` is `actual - predicted` — recall `02-Linear Regression/theory.md` Section 4, this is exactly the **residual** the whole training process was built to minimize (in aggregate, via MSE) on the *training* set; here we're seeing what those residuals actually look like on genuinely new data.

---
## 8. Model Evaluation

Same principle as Chapter 3: a number like "R² = 0.67" means little alone — always compare against a baseline, and use multiple metrics, not just one.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# sklearn.metrics: a module of ready-made scoring functions -- each one takes (actual_values, predicted_values) and returns a single number.

# Baseline: what if we just always predicted the TRAINING MEAN mpg, ignoring every feature?
baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
# np.full_like(y_test, value) makes a new array the SAME SHAPE as y_test, but filled entirely with one repeated value (here, the training mean)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
# mean_squared_error(actual, predicted) computes MSE (theory.md Section 4) in one call; np.sqrt() undoes the squaring to get RMSE, in real MPG units

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)          # theory.md Section 9 -- undoes MSE's squaring, back into real MPG units
mae = mean_absolute_error(y_test, y_pred)   # mean_absolute_error: average of |actual - predicted| -- no squaring, so big misses aren't punished extra hard
r2 = r2_score(y_test, y_pred)                # r2_score: computes R² (theory.md Section 9) -- fraction of MPG's variation the model explains, 0 to 1

print(f"Baseline RMSE (always predict the mean): {baseline_rmse:.3f}")
print(f"Model RMSE:                              {rmse:.3f}")
print(f"Model MAE:                               {mae:.3f}")
print(f"Model R²:                                {r2:.3f}")
# Baseline RMSE (always predict the mean): 10.163
# Model RMSE:                              5.654
# Model MAE:                               4.114
# Model R²:                                0.675


**Reading this:**
- **RMSE dropped from 10.16 (baseline) to 5.65 (model)** — on average, the model's predictions are off by about 5.65 MPG, roughly half the error of just guessing the average every time. A real, meaningful improvement.
- **MAE (4.11)** — the average *absolute* error, in the same MPG units, slightly more intuitive and less sensitive to any single large miss than RMSE (recall theory.md: squaring punishes big misses harder — MAE skips that squaring).
- **R² = 0.675** — the model explains about 67.5% of the variation in MPG across cars. Not perfect (32.5% of the variation is unexplained by these 4 features alone), but a genuinely useful model, not a coin flip.

In [ ]:
plt.figure(figsize=(5.5, 5.5))   # starts a new, single figure (as opposed to Step 3's 2x2 grid) sized 5.5x5.5 inches
plt.scatter(y_test, y_pred, alpha=0.7)   # one dot per test-set car: x = actual MPG, y = predicted MPG; alpha=0.7 makes dots slightly translucent (helps see overlaps)
lims = [min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2]
# lims: a shared [min, max] range covering both actual and predicted values (with a little padding), so the diagonal line below spans the whole plot
plt.plot(lims, lims, "r--", label="Perfect prediction line")   # plots a straight diagonal line (x=y) -- "r--" means red, dashed
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")
plt.title("Actual vs. Predicted MPG (test set)")
plt.legend()          # shows the "Perfect prediction line" label as a legend box
plt.tight_layout()
plt.show()


**Reading this plot:** if every prediction were perfect, every point would sit exactly on the red dashed line. Points scattered around the line (not on it) are expected — that scatter *is* the residual error from Step 7, visualized. Points far from the line are where the model struggled most.

### Interpreting the coefficients — and the multicollinearity story from Step 3

Recall Step 3: every feature *individually* correlated **negatively** with MPG — more horsepower, volume, speed, or weight each alone associated with lower MPG. Compare that to what the trained model's coefficients (Step 6) actually say:

| Feature | Solo correlation with MPG | Multivariate coefficient (m) |
|---|---|---|
| HP | -0.725 | **-0.221** (same direction) |
| VOL | -0.529 | **-0.540** (same direction) |
| SP | -0.687 | **+0.534** (flipped!) |
| WT | -0.527 | **+0.945** (flipped!) |

**`SP` and `WT` flipped sign** between "on their own" and "combined with the others." This is the multicollinearity warning from Step 3 showing its real effect: because `VOL`≈`WT` and `HP`≈`SP` are each carrying almost duplicate information, the model can't cleanly assign credit to each one individually — it ends up distributing (and sometimes flipping) their effects in a way that's mathematically valid for *prediction*, but **not trustworthy for individual interpretation**.

**The practical lesson:** this model's *predictions* (Step 8's RMSE/R²) are still meaningful and usable. But you should **not** walk away saying "increasing weight *causes* better MPG" just because `WT`'s coefficient is positive here — that would be reading a multicollinearity artifact as a real business insight. This is exactly why `theory.md` Section 8 flags multicollinearity as something to check — a real next step here would be dropping `VOL` or `WT` (keep only one of the pair) and refitting, to get coefficients that are actually safe to interpret individually.

---
## 9. Model Deployment

"Deployment" means taking the trained model out of this notebook and saving it as a standalone file that any other program (a web app, an API, a UI) can load and use to make predictions — without needing this notebook, the training data, or any of the training code ever again. This is the parametric idea from Chapter 1 made completely literal: the saved file is just the learned `m`'s and `b`, nothing more.

In [ ]:
import joblib
# joblib: the standard library for saving/loading Python objects to disk -- works especially well for sklearn models
# (it's optimized for objects containing large numpy arrays, which is exactly what a trained model's m's/b are stored as internally).

joblib.dump(model, "linear_regression_mpg_model.pkl")
# joblib.dump(object, filepath) serializes the ENTIRE trained model object (architecture + learned m's/b) into one file on disk.
print("Model saved to linear_regression_mpg_model.pkl")


In [ ]:
# Simulate what a separate application (a UI, an API) would do: load the file fresh
# and use it to predict on a brand-new car it's never seen -- no access to training
# data or this notebook needed, just the saved file.
loaded_model = joblib.load("linear_regression_mpg_model.pkl")
# joblib.load(filepath) reconstructs the exact trained model object from the saved file -- loaded_model behaves IDENTICALLY to the original `model`.

new_car = pd.DataFrame({"HP": [120], "VOL": [95], "SP": [115], "WT": [30]})
# Building a 1-row DataFrame by hand, with the SAME column names/order the model was trained on -- required for .predict() to work correctly.
predicted_mpg = loaded_model.predict(new_car)   # returns an array with ONE value in it (since we gave it one row)
print(f"Predicted MPG for a new car: {predicted_mpg[0]:.2f}")   # [0] pulls that single value out of the array to print it as a plain number


**What happens after this, in a real project (not built here):** this `.pkl` file would get wrapped in a small API (e.g. a Flask/FastAPI endpoint that loads the file once and exposes a `/predict` route), which a UI (a web form, a mobile app) calls whenever a user wants a prediction. That API + UI wiring is typically where a **Data Scientist's job ends and an ML/Software Engineer's job continues** — the DS is responsible for the model being *correct and well-evaluated* (Steps 1-8), not usually for building/scaling the serving infrastructure itself.

---
## 10. CI/CD (Continuous Integration / Continuous Deployment)

This step isn't something we can meaningfully *run* inside a notebook — it's an engineering practice that wraps around the whole pipeline once it's a real, ongoing system, not a one-off notebook run. Explained conceptually:

- **CI (Continuous Integration):** whenever the training code or data changes, automatically re-run tests (does the pipeline still run end-to-end without errors? does the retrained model still meet a minimum quality bar, e.g. R² above some threshold?) before anything gets merged/accepted.
- **CD (Continuous Deployment):** once a new model passes those checks, automatically package it (Step 9's `.pkl` file, or similar) and push it live to replace the old one — without a human manually repeating Steps 1-9 by hand every time.

**Why this matters, tying back to your earlier question about model drift:** a model trained once and never touched again slowly gets worse as the real world changes (new car designs, new driving patterns, whatever the domain is) — this is **model drift**. CI/CD is the machinery that makes it *cheap and safe* to retrain and reship a model regularly, instead of it being a manual, error-prone, "someone has to remember to do this" process. In a mature ML team, Steps 1-9 above might run automatically on a schedule (say, weekly, on fresh data) with CI/CD handling the testing and reshipping — a human mainly monitors and intervenes when something looks wrong, rather than re-running every step by hand each time.

---
## Recap — the 10 steps, what we actually did at each one

1. **Business Understanding** — predict MPG from HP/VOL/SP/WT; a regression problem since MPG is a number.
2. **Data Collection** — `Cars.csv`, already provided.
3. **Data Understanding** — no missing values, but found a real multicollinearity red flag (VOL≈WT, HP≈SP) before modeling anything.
4. **Data Preparation** — separated features/target, train/test split (no `stratify` needed — that's a classification-only concept).
5. **Model Building** — chose and instantiated `LinearRegression` (untrained at this point).
6. **Model Training** — `.fit()` — OLS solves directly for `m`'s and `b`.
7. **Model Testing** — generated predictions on the held-out test set, looked at raw actual-vs-predicted rows.
8. **Model Evaluation** — RMSE/MAE/R² vs. a mean-prediction baseline, an actual-vs-predicted plot, and — importantly — traced Step 3's multicollinearity warning through to real sign-flips in the coefficients, with an honest caveat about what's safe to interpret vs. not.
9. **Model Deployment** — saved the trained model as a `.pkl` file, and proved a fresh process can load it and predict without the original notebook/data, then flagged where DS work typically hands off to engineering.
10. **CI/CD** — explained conceptually (not run here): the automation that makes retraining/reshipping safe and routine as real-world data drifts, rather than a manual one-off process.